## Broadcast joins & join performance

A normal join **shuffles both sides** by the join key across the cluster — expensive once the tables are large. A **broadcast join** ships the *smaller* side to every executor, so the big side never moves. No shuffle, no network storm.

**Spark auto-broadcasts** when one side's estimated size is below `spark.sql.autoBroadcastJoinThreshold` — default **10 MB**, often tuned to 100 MB+. Joining 50 billion `card_transactions` rows against a 5 MB `merchant_categories` lookup auto-broadcasts.

**Force a broadcast** when stats are stale or you know better than the optimiser:

```python
from pyspark.sql.functions import broadcast
df.join(broadcast(small_df), on="merchant_category", how="left")
```

```sql
SELECT /*+ BROADCAST(c) */ t.*, c.category_group
FROM card_transactions t
LEFT JOIN merchant_categories c ON t.merchant_category = c.code
```

**When it backfires:** broadcasting a side larger than driver memory blows the driver. Cap the threshold at the order of the largest small table you actually have.

**Adaptive Query Execution** (AQE — covered in module 08) can promote a sort-merge join to a broadcast join **at runtime**, once it sees real partition stats, even if planner statistics said otherwise. So the two levers on join performance are: **shrink the shuffle** (broadcast the small side) and **let AQE re-plan** on actual data.
